In [1]:
"""
RobustCIFAR Classifier — 2-Seed Ensemble + Label Propagation + Submission
===================================================================
End-to-end pipeline: trains a 2-seed ensemble from scratch, propagates
high-confidence soft labels to unlabeled test data, refines, then produces
predictions.
"""

import os
import sys
import csv
import time
import math
import copy
import random
import argparse
from collections import Counter

import numpy as np
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler, Subset
from torch.optim.swa_utils import AveragedModel, SWALR
from torchvision import transforms

# ── SETTINGS ─────────────────────────────────────────────────────────────────

DATASET_PATH = "shift-guard-10-robust-image-classification-challenge"
RESULTS_DIR  = "."

LABEL_LIST = [
    "airplane", "automobile", "bird", "cat", "deer",
    "dog", "frog", "horse", "ship", "truck"
]
LABEL_TO_INT = {name: i for i, name in enumerate(LABEL_LIST)}
INT_TO_LABEL = {i: name for name, i in LABEL_TO_INT.items()}
N_CATEGORIES = 10

IMG_MEAN = (0.4914, 0.4822, 0.4465)
IMG_STD  = (0.2470, 0.2435, 0.2616)


# ── UTILITIES ─────────────────────────────────────────────────────────────────

def fix_random_seeds(s):
    random.seed(s)
    np.random.seed(s)
    torch.manual_seed(s)
    torch.cuda.manual_seed_all(s)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


class RunningMetric:
    def __init__(self):
        self.val = self.avg = self.total = self.n = 0

    def update(self, v, k=1):
        self.val = v
        self.total += v * k
        self.n += k
        self.avg = self.total / self.n


def calc_f1(pred_list, target_list):
    from sklearn.metrics import f1_score
    return f1_score(target_list, pred_list, average="macro", zero_division=0)


# ── AUGMENTATION ──────────────────────────────────────────────────────────────

class PatchEraser:
    """Zeros out a random square region of the input tensor."""
    def __init__(self, patch=16):
        self.patch = patch

    def __call__(self, t):
        h, w = t.size(1), t.size(2)
        mask = torch.ones(h, w, dtype=t.dtype)
        cy = random.randint(0, h - 1)
        cx = random.randint(0, w - 1)
        r0 = max(0, cy - self.patch // 2)
        r1 = min(h, cy + self.patch // 2)
        c0 = max(0, cx - self.patch // 2)
        c1 = min(w, cx + self.patch // 2)
        mask[r0:r1, c0:c1] = 0.0
        return t * mask.unsqueeze(0)


def build_train_aug():
    return transforms.Compose([
        transforms.RandomCrop(32, padding=4, fill=128),
        transforms.RandomHorizontalFlip(),
        transforms.AutoAugment(transforms.AutoAugmentPolicy.CIFAR10),
        transforms.ToTensor(),
        transforms.Normalize(IMG_MEAN, IMG_STD),
        PatchEraser(patch=16),
    ])


def build_eval_aug():
    return transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(IMG_MEAN, IMG_STD),
    ])


def build_tta_aug():
    return transforms.Compose([
        transforms.RandomCrop(32, padding=4, fill=128),
        transforms.RandomHorizontalFlip(),
        transforms.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1),
        transforms.RandomRotation(10, fill=128),
        transforms.ToTensor(),
        transforms.Normalize(IMG_MEAN, IMG_STD),
    ])


# ── DATASET ───────────────────────────────────────────────────────────────────

class RobustImageDataset(Dataset):
    def __init__(self, root, split="train", transform=None,
                 holdout=0.05, rng_seed=42, soft_labels=None):
        self.root      = root
        self.split     = split
        self.transform = transform
        self.records   = []

        if split in ("train", "val"):
            label_file = os.path.join(root, "train_labels.csv")
            file_ids, raw_labels = [], []
            with open(label_file) as fh:
                for row in csv.DictReader(fh):
                    file_ids.append(row["id"].strip().zfill(6))
                    raw_labels.append(row["label"].strip())

            rng = np.random.RandomState(rng_seed)
            per_class_idx = {c: [] for c in LABEL_LIST}
            for i, lbl in enumerate(raw_labels):
                per_class_idx[lbl].append(i)

            trn_indices, val_indices = [], []
            for c in LABEL_LIST:
                bucket = per_class_idx[c][:]
                rng.shuffle(bucket)
                n_hold = max(1, int(len(bucket) * holdout))
                val_indices.extend(bucket[:n_hold])
                trn_indices.extend(bucket[n_hold:])

            chosen = trn_indices if split == "train" else val_indices
            for i in chosen:
                fp = os.path.join(root, "train_images", f"{file_ids[i]}.png")
                self.records.append((fp, LABEL_TO_INT[raw_labels[i]]))

            if split == "train" and soft_labels is not None:
                for img_id, cls_idx in soft_labels:
                    fp = os.path.join(root, "test_images", f"{img_id}.png")
                    self.records.append((fp, cls_idx))

        elif split == "test":
            sub_file = os.path.join(root, "sample_submission.csv")
            with open(sub_file) as fh:
                for row in csv.DictReader(fh):
                    uid = row["id"].strip().zfill(6)
                    fp  = os.path.join(root, "test_images", f"{uid}.png")
                    self.records.append((fp, uid))

    def __len__(self):
        return len(self.records)

    def __getitem__(self, i):
        fp, tgt = self.records[i]
        img = Image.open(fp).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, tgt

    def class_histogram(self):
        if self.split == "test":
            return None
        cats = [r[1] for r in self.records]
        return np.bincount(cats, minlength=N_CATEGORIES)

    def weighted_sampler(self):
        cats  = [r[1] for r in self.records]
        freq  = np.bincount(cats, minlength=N_CATEGORIES)
        inv_sqrt = 1.0 / (np.sqrt(freq) + 1e-6)
        weights  = [inv_sqrt[c] for c in cats]
        return WeightedRandomSampler(weights, len(cats), replacement=True)


# ── MODEL: Wide Residual Network ──────────────────────────────────────────────

class ResidualUnit(nn.Module):
    def __init__(self, c_in, c_out, stride, drop=0.3):
        super().__init__()
        self.norm1 = nn.BatchNorm2d(c_in)
        self.proj1 = nn.Conv2d(c_in, c_out, 3, stride=stride, padding=1, bias=False)
        self.norm2 = nn.BatchNorm2d(c_out)
        self.proj2 = nn.Conv2d(c_out, c_out, 3, stride=1, padding=1, bias=False)
        self.drop  = nn.Dropout(p=drop) if drop > 0 else nn.Identity()
        self.skip  = nn.Sequential()
        if stride != 1 or c_in != c_out:
            self.skip = nn.Conv2d(c_in, c_out, 1, stride=stride, bias=False)

    def forward(self, x):
        h = self.proj1(F.relu(self.norm1(x), inplace=True))
        h = self.drop(h)
        h = self.proj2(F.relu(self.norm2(h), inplace=True))
        return h + self.skip(x)


class BroadResNet(nn.Module):
    """Wide residual network for 32x32 inputs (~36M params at depth=28, k=10)."""
    def __init__(self, depth=28, width=10, n_cls=10, drop=0.3):
        super().__init__()
        assert (depth - 4) % 6 == 0
        n    = (depth - 4) // 6
        dims = [16, 16 * width, 32 * width, 64 * width]
        self.stem   = nn.Conv2d(3, dims[0], 3, stride=1, padding=1, bias=False)
        self.stage1 = self._block(dims[0], dims[1], n, 1, drop)
        self.stage2 = self._block(dims[1], dims[2], n, 2, drop)
        self.stage3 = self._block(dims[2], dims[3], n, 2, drop)
        self.norm   = nn.BatchNorm2d(dims[3])
        self.head   = nn.Linear(dims[3], n_cls)
        self._reset_params()

    def _block(self, ci, co, n, stride, drop):
        units = [ResidualUnit(ci, co, stride, drop)]
        for _ in range(1, n):
            units.append(ResidualUnit(co, co, 1, drop))
        return nn.Sequential(*units)

    def _reset_params(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode="fan_out", nonlinearity="relu")
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.ones_(m.weight)
                nn.init.zeros_(m.bias)

    def forward(self, x):
        z = self.stem(x)
        z = self.stage1(z)
        z = self.stage2(z)
        z = self.stage3(z)
        z = F.relu(self.norm(z), inplace=True)
        z = F.adaptive_avg_pool2d(z, 1).flatten(1)
        return self.head(z)


# ── LOSS ──────────────────────────────────────────────────────────────────────

class FrequencyAdjustedLoss(nn.Module):
    """Cross-entropy that subtracts log-class-prior to counter class imbalance."""
    def __init__(self, freq_counts, smoothing=0.0):
        super().__init__()
        p = torch.tensor(freq_counts, dtype=torch.float32)
        p = p / p.sum()
        self.register_buffer("log_prior", torch.log(p + 1e-12))
        self.smoothing = smoothing

    def forward(self, logits, y):
        adj = logits + self.log_prior.unsqueeze(0)
        return F.cross_entropy(adj, y, label_smoothing=self.smoothing)


# ── DATA MIXING ───────────────────────────────────────────────────────────────

def blend_samples(x, y, alpha=1.0):
    ratio = np.random.beta(alpha, alpha) if alpha > 0 else 1.0
    perm  = torch.randperm(x.size(0), device=x.device)
    return ratio * x + (1 - ratio) * x[perm], y, y[perm], ratio


def patch_mix(x, y, alpha=1.0):
    ratio  = np.random.beta(alpha, alpha) if alpha > 0 else 1.0
    perm   = torch.randperm(x.size(0), device=x.device)
    _, _, H, W = x.shape
    side_w = int(W * math.sqrt(1.0 - ratio))
    side_h = int(H * math.sqrt(1.0 - ratio))
    ox = random.randint(0, W - 1)
    oy = random.randint(0, H - 1)
    c0 = max(0, ox - side_w // 2)
    c1 = min(W, ox + side_w // 2)
    r0 = max(0, oy - side_h // 2)
    r1 = min(H, oy + side_h // 2)
    out = x.clone()
    out[:, :, r0:r1, c0:c1] = x[perm, :, r0:r1, c0:c1]
    actual_ratio = 1 - (r1 - r0) * (c1 - c0) / (H * W)
    return out, y, y[perm], actual_ratio


def blended_loss(criterion, logits, ya, yb, lam):
    return lam * criterion(logits, ya) + (1 - lam) * criterion(logits, yb)


# ── TRAINING ENGINE ───────────────────────────────────────────────────────────

def run_train_epoch(net, loader, criterion, opt, scaler, device, aug_prob=0.5):
    net.train()
    loss_tracker = RunningMetric()
    correct = total = 0

    for imgs, lbls in loader:
        imgs, lbls = imgs.to(device), lbls.to(device)
        mixed = False
        opt.zero_grad()

        with torch.amp.autocast('cuda'):
            if aug_prob > 0 and random.random() < aug_prob:
                fn = blend_samples if random.random() < 0.5 else patch_mix
                imgs, ya, yb, lam = fn(imgs, lbls, 1.0)
                out  = net(imgs)
                loss = blended_loss(criterion, out, ya, yb, lam)
                mixed = True
            else:
                out  = net(imgs)
                loss = criterion(out, lbls)

        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(net.parameters(), max_norm=5.0)
        scaler.step(opt)
        scaler.update()

        loss_tracker.update(loss.item(), imgs.size(0))
        if not mixed:
            _, hat = out.max(1)
            correct += hat.eq(lbls).sum().item()
            total   += lbls.size(0)

    return loss_tracker.avg, 100.0 * correct / total if total > 0 else 0.0


@torch.no_grad()
def evaluate(net, loader, criterion, device):
    net.eval()
    loss_tracker = RunningMetric()
    pred_list, target_list = [], []

    for imgs, lbls in loader:
        imgs, lbls = imgs.to(device), lbls.to(device)
        out  = net(imgs)
        loss = criterion(out, lbls)
        loss_tracker.update(loss.item(), imgs.size(0))
        _, hat = out.max(1)
        pred_list.extend(hat.cpu().numpy())
        target_list.extend(lbls.cpu().numpy())

    score = calc_f1(pred_list, target_list)
    acc   = 100.0 * np.mean(np.array(pred_list) == np.array(target_list))
    return loss_tracker.avg, acc, score


def fit_model(seed, cfg, device, freq_counts):
    fix_random_seeds(seed)
    print(f"\n{'='*60}")
    print(f"  Phase 1 Training | Seed={seed} | Epochs={cfg.epochs}")
    print(f"{'='*60}\n")

    trn_ds = RobustImageDataset(DATASET_PATH, "train", build_train_aug(),
                                holdout=cfg.val_ratio, rng_seed=42)
    val_ds = RobustImageDataset(DATASET_PATH, "val",   build_eval_aug(),
                                holdout=cfg.val_ratio, rng_seed=42)

    sampler    = trn_ds.weighted_sampler()
    trn_loader = DataLoader(trn_ds, batch_size=cfg.batch_size, shuffle=False,
                            sampler=sampler, num_workers=4,
                            pin_memory=True, drop_last=True)
    val_loader = DataLoader(val_ds, batch_size=cfg.batch_size * 2,
                            shuffle=False, num_workers=4, pin_memory=True)

    net       = BroadResNet(depth=28, width=10, n_cls=N_CATEGORIES, drop=0.3).to(device)
    smooth    = 0.0 if cfg.mix_prob > 0 else cfg.label_smoothing
    criterion = FrequencyAdjustedLoss(freq_counts, smoothing=smooth).to(device)
    opt       = torch.optim.AdamW(net.parameters(), lr=cfg.lr,
                                  weight_decay=cfg.wd, eps=1e-4)

    ramp_epochs = 5
    def _sched(ep):
        if ep < ramp_epochs:
            return (ep + 1) / ramp_epochs
        prog = (ep - ramp_epochs) / (cfg.epochs - ramp_epochs)
        return 0.5 * (1 + math.cos(math.pi * prog))
    sched = torch.optim.lr_scheduler.LambdaLR(opt, _sched)

    avg_start = cfg.swa_start
    do_avg    = avg_start < cfg.epochs
    if do_avg:
        avg_model = AveragedModel(net)
        avg_sched = SWALR(opt, swa_lr=cfg.swa_lr)

    scaler       = torch.amp.GradScaler('cuda')
    top_f1       = 0.0
    best_weights = None

    for ep in range(cfg.epochs):
        t0 = time.time()
        tr_loss, tr_acc = run_train_epoch(net, trn_loader, criterion,
                                          opt, scaler, device, cfg.mix_prob)

        if do_avg and ep >= avg_start:
            avg_model.update_parameters(net)
            avg_sched.step()
        else:
            sched.step()

        vl_loss, vl_acc, vl_f1 = evaluate(net, val_loader, criterion, device)
        tag = " [SWA]" if (do_avg and ep >= avg_start) else ""
        print(f"  E{ep+1:3d}/{cfg.epochs} | TrL:{tr_loss:.3f} TrA:{tr_acc:.1f}% "
              f"| VL:{vl_loss:.3f} VA:{vl_acc:.1f}% F1:{vl_f1:.4f} "
              f"| {time.time()-t0:.1f}s{tag}")

        if vl_f1 > top_f1:
            top_f1       = vl_f1
            best_weights = copy.deepcopy(net.state_dict())

    if do_avg:
        torch.optim.swa_utils.update_bn(trn_loader, avg_model, device=device)
        _, _, vl_f1 = evaluate(avg_model, val_loader, criterion, device)
        if vl_f1 > top_f1:
            top_f1       = vl_f1
            best_weights = copy.deepcopy(avg_model.module.state_dict())

    return best_weights, top_f1


def refine_with_pseudolabels(model_weights, soft_labels, freq_counts, cfg, device):
    """Phase 2: re-trains each model with high-confidence test samples appended."""
    print(f"\n{'='*60}")
    print(f"  Phase 2: Fine-Tuning with {len(soft_labels)} Pseudo-Labels")
    print(f"{'='*60}\n")

    trn_ds = RobustImageDataset(DATASET_PATH, "train", build_train_aug(),
                                holdout=cfg.val_ratio, rng_seed=42,
                                soft_labels=soft_labels)
    val_ds = RobustImageDataset(DATASET_PATH, "val", build_eval_aug(),
                                holdout=cfg.val_ratio, rng_seed=42)

    sampler    = trn_ds.weighted_sampler()
    trn_loader = DataLoader(trn_ds, batch_size=cfg.batch_size, shuffle=False,
                            sampler=sampler, num_workers=4,
                            pin_memory=True, drop_last=True)
    val_loader = DataLoader(val_ds, batch_size=cfg.batch_size * 2,
                            shuffle=False, num_workers=4, pin_memory=True)

    refined = []
    scaler  = torch.amp.GradScaler('cuda')

    for idx, w in enumerate(model_weights):
        print(f"\n  Fine-Tuning Model {idx+1}/{len(model_weights)} "
              f"(Seed {cfg.seeds[idx]})...")
        net       = BroadResNet(depth=28, width=10, n_cls=N_CATEGORIES, drop=0.3).to(device)
        net.load_state_dict(w)
        opt       = torch.optim.AdamW(net.parameters(), lr=1e-4,
                                      weight_decay=cfg.wd, eps=1e-4)
        criterion = FrequencyAdjustedLoss(freq_counts, smoothing=0.0).to(device)
        top_f1       = 0.0
        best_weights = copy.deepcopy(net.state_dict())

        for ep in range(cfg.ft_epochs):
            tr_loss, tr_acc = run_train_epoch(net, trn_loader, criterion,
                                              opt, scaler, device, cfg.mix_prob)
            vl_loss, vl_acc, vl_f1 = evaluate(net, val_loader, criterion, device)
            print(f"    FT E{ep+1:2d}/{cfg.ft_epochs} | TrL:{tr_loss:.3f} "
                  f"TrA:{tr_acc:.1f}% | VL:{vl_loss:.3f} VA:{vl_acc:.1f}% "
                  f"F1:{vl_f1:.4f}")
            if vl_f1 > top_f1:
                top_f1       = vl_f1
                best_weights = copy.deepcopy(net.state_dict())

        refined.append(best_weights)

    return refined


# ── INFERENCE ─────────────────────────────────────────────────────────────────

@torch.no_grad()
def infer_ensemble_views(net, test_ds, device, n_aug=12, bsz=512):
    net.eval()
    test_ds.transform = build_eval_aug()
    loader = DataLoader(test_ds, batch_size=bsz, shuffle=False,
                        num_workers=4, pin_memory=True)

    prob_list, uid_list = [], []
    for imgs, uids in loader:
        p = F.softmax(net(imgs.to(device)), dim=1)
        prob_list.append(p.cpu())
        uid_list.extend(uids)
    running = torch.cat(prob_list, dim=0)

    for _ in range(n_aug):
        test_ds.transform = build_tta_aug()
        loader = DataLoader(test_ds, batch_size=bsz, shuffle=False,
                            num_workers=4, pin_memory=True)
        vp = []
        for imgs, _ in loader:
            vp.append(F.softmax(net(imgs.to(device)), dim=1).cpu())
        running += torch.cat(vp, dim=0)

    return running / (n_aug + 1), uid_list


def write_predictions(probs, uid_list, out_path):
    cls_idx     = probs.argmax(dim=1).numpy()
    label_names = [INT_TO_LABEL[c] for c in cls_idx]
    with open(out_path, "w", newline="") as fh:
        fh.write("id,label\n")
        for uid, lbl in zip(uid_list, label_names):
            fh.write(f"{uid},{lbl}\n")
    print(f"\n  Submission saved: {out_path}")


# ── MAIN ──────────────────────────────────────────────────────────────────────

def main():
    ap = argparse.ArgumentParser(description="RobustCIFAR classifier")
    ap.add_argument("--debug",            action="store_true")
    ap.add_argument("--gpu",              type=int,   default=0)
    ap.add_argument("--data-root",        type=str,   default=None)
    ap.add_argument("--epochs",           type=int,   default=225)
    ap.add_argument("--swa-start",        type=int,   default=165)
    ap.add_argument("--seeds",            type=int,   nargs="+", default=[42, 137])
    ap.add_argument("--ft-epochs",        type=int,   default=15)
    ap.add_argument("--pseudo-threshold", type=float, default=0.90)
    ap.add_argument("--batch-size",       type=int,   default=128)
    ap.add_argument("--lr",               type=float, default=3e-3)
    ap.add_argument("--wd",               type=float, default=1e-4)
    ap.add_argument("--val-ratio",        type=float, default=0.05)
    ap.add_argument("--swa-lr",           type=float, default=5e-4)
    ap.add_argument("--tta",              type=int,   default=12)
    ap.add_argument("--mix-prob",         type=float, default=0.5)
    ap.add_argument("--label-smoothing",  type=float, default=0.1)
    ap.add_argument("--output",           type=str,   default="submission.csv")
    cfg, _ = ap.parse_known_args()

    global DATASET_PATH
    if cfg.data_root:
        DATASET_PATH = cfg.data_root
    else:
        for p in [
            "/kaggle/input/competitions/shift-guard-10-robust-image-classification-challenge",
            "/kaggle/input/shift-guard-10-robust-image-classification-challenge",
            "shift-guard-10-robust-image-classification-challenge",
            os.path.join(os.getcwd(), "shift-guard-10-robust-image-classification-challenge"),
        ]:
            if os.path.isdir(p):
                DATASET_PATH = p
                break

    if not os.path.isdir(DATASET_PATH):
        print(f"ERROR: Data not found at: {DATASET_PATH}")
        sys.exit(1)

    device = torch.device(f"cuda:{cfg.gpu}" if torch.cuda.is_available() else "cpu")
    tmp = RobustImageDataset(DATASET_PATH, "train", holdout=cfg.val_ratio, rng_seed=42)
    freq_counts = tmp.class_histogram()
    del tmp

    # ── Phase 1: initial training ─────────────────────────────────────────────
    model_weights = []
    for s in cfg.seeds:
        w, f1 = fit_model(s, cfg, device, freq_counts)
        model_weights.append(w)

    # ── Generate soft labels ──────────────────────────────────────────────────
    print(f"\n{'='*60}")
    print(f"  Generating Predictions for Pseudo-Labeling Filtering...")
    print(f"{'='*60}")

    test_ds = RobustImageDataset(DATASET_PATH, "test", build_eval_aug())
    combined_probs = None

    for w in model_weights:
        net = BroadResNet(depth=28, width=10, n_cls=N_CATEGORIES, drop=0.3).to(device)
        net.load_state_dict(w)
        p, uid_list = infer_ensemble_views(net, test_ds, device,
                                           n_aug=cfg.tta, bsz=512)
        combined_probs = p if combined_probs is None else combined_probs + p
        del net
        torch.cuda.empty_cache()

    combined_probs /= len(model_weights)
    conf, hat = combined_probs.max(dim=1)
    soft_labels = [(uid_list[i], hat[i].item())
                   for i, c in enumerate(conf) if c.item() > cfg.pseudo_threshold]

    # ── Phase 2: refine + final inference ────────────────────────────────────
    if soft_labels:
        model_weights = refine_with_pseudolabels(
            model_weights, soft_labels, freq_counts, cfg, device)

        print(f"\n{'='*60}")
        print(f"  Generating Final Submission...")
        print(f"{'='*60}")
        combined_probs = None
        for w in model_weights:
            net = BroadResNet(depth=28, width=10, n_cls=N_CATEGORIES, drop=0.3).to(device)
            net.load_state_dict(w)
            p, uid_list = infer_ensemble_views(net, test_ds, device,
                                               n_aug=cfg.tta, bsz=512)
            combined_probs = p if combined_probs is None else combined_probs + p
            del net
            torch.cuda.empty_cache()
        combined_probs /= len(model_weights)

    out_path = os.path.join(RESULTS_DIR, cfg.output)
    write_predictions(combined_probs, uid_list, out_path)


if __name__ == "__main__":
    main()



  Phase 1 Training | Seed=42 | Epochs=225

  E  1/225 | TrL:1.937 TrA:6.9% | VL:1.406 VA:17.8% F1:0.1476 | 82.3s
  E  2/225 | TrL:1.840 TrA:11.0% | VL:1.672 VA:14.0% F1:0.1274 | 87.8s
  E  3/225 | TrL:1.714 TrA:15.3% | VL:2.003 VA:20.3% F1:0.1579 | 87.3s
  E  4/225 | TrL:1.674 TrA:18.9% | VL:3.242 VA:11.7% F1:0.1199 | 87.0s
  E  5/225 | TrL:1.647 TrA:21.3% | VL:1.392 VA:40.9% F1:0.3242 | 86.6s
  E  6/225 | TrL:1.563 TrA:25.1% | VL:1.931 VA:35.0% F1:0.3087 | 86.5s
  E  7/225 | TrL:1.519 TrA:28.3% | VL:1.471 VA:33.0% F1:0.2959 | 86.4s
  E  8/225 | TrL:1.507 TrA:30.3% | VL:1.384 VA:43.5% F1:0.3765 | 86.7s
  E  9/225 | TrL:1.445 TrA:32.7% | VL:1.054 VA:49.3% F1:0.4212 | 86.4s
  E 10/225 | TrL:1.424 TrA:35.4% | VL:1.020 VA:56.7% F1:0.4570 | 86.8s
  E 11/225 | TrL:1.392 TrA:37.9% | VL:1.260 VA:50.5% F1:0.4137 | 86.7s
  E 12/225 | TrL:1.382 TrA:39.8% | VL:1.055 VA:51.1% F1:0.4536 | 86.1s
  E 13/225 | TrL:1.336 TrA:41.7% | VL:0.894 VA:51.1% F1:0.4259 | 86.0s
  E 14/225 | TrL:1.346 TrA:43.2% |